# 01 — Landmark Extraction

Her video için MediaPipe Holistic ile landmark çıkarır.

| Parametre | Değer |
|-----------|-------|
| Video boyutu | 512×512 (orijinal, küçültme yok) |
| Frame sayısı | 16 per video |
| Landmark boyutu | 225 per video (sol_el 63 + sağ_el 63 + pose 99) |
| Color + Depth concat | **450** per frame |
| Output shape | **(16, 450)** per sample |
| GPU gereksinimi | ❌ CPU yeterli |

**Resume desteği var** — Colab kopsa bile kaldığı yerden devam eder.

In [ ]:
!pip install mediapipe==0.10.9 -q
!pip install opencv-python-headless -q
print('Kurulum tamam')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, cv2, time, traceback
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm.notebook import tqdm

# ═══════════════════════════════════════════════════
# CONFIG — değiştirmen gereken tek şey BASE path
# ═══════════════════════════════════════════════════
BASE        = "/content/drive/MyDrive/sign-language-project"
DATASET     = f"{BASE}/dataset"
TRAIN_DIR   = f"{DATASET}/train"
VAL_DIR     = f"{DATASET}/validation"
TRAIN_CSV   = f"{DATASET}/train_labels.csv"
VAL_CSV     = f"{DATASET}/validation_labels.csv"
TRAIN_CACHE = f"{DATASET}/cached_landmarks/train"
VAL_CACHE   = f"{DATASET}/cached_landmarks/validation"
PACKED_DIR  = f"{BASE}/packed_landmarks"

MAX_FRAMES    = 16
IMG_SIZE      = 512   # Orijinal video boyutu — küçültme yok, kalite kayıpsız
SINGLE_LM_DIM = 225   # Tek video: sol_el(63) + sağ_el(63) + pose(99)
FINAL_DIM     = 450   # Color(225) + Depth(225) concat

os.makedirs(TRAIN_CACHE, exist_ok=True)
os.makedirs(VAL_CACHE,   exist_ok=True)
os.makedirs(PACKED_DIR,  exist_ok=True)

print('Config OK')
print(f'Her örnek output shape: ({MAX_FRAMES}, {FINAL_DIM})')
print(f'Train cache klasörü: {TRAIN_CACHE}')
print(f'Val cache klasörü:   {VAL_CACHE}')

In [ ]:
# ═══════════════════════════════════════════════════
# CSV YÜKLE VE DOĞRULA
# ═══════════════════════════════════════════════════
train_df = pd.read_csv(TRAIN_CSV, header=None, names=['sample_id', 'label'])
val_df   = pd.read_csv(VAL_CSV,   header=None, names=['sample_id', 'label'])

print(f'Train: {len(train_df)} satır, {train_df["label"].nunique()} unique label')
print(f'Val:   {len(val_df)} satır, {val_df["label"].nunique()} unique label')
print(f'Label aralığı: {train_df["label"].min()} - {train_df["label"].max()}')
print(f'\nÖrnek satırlar:')
print(train_df.head(3).to_string())

# Temel kontrol
assert train_df['label'].min() == 0, 'Label 0dan başlamıyor!'
assert train_df['label'].max() == 225, 'Max label 225 değil!'
print('\n✅ CSV formatı doğru')

In [ ]:
# ═══════════════════════════════════════════════════
# VİDEO VARLIK KONTROLÜ
# ═══════════════════════════════════════════════════
# Kaç color/depth video gerçekten var?
import random

def check_video_availability(df, video_dir, n_sample=200):
    sample_ids = df['sample_id'].tolist()
    check_ids = random.sample(sample_ids, min(n_sample, len(sample_ids)))
    
    color_found = depth_found = both_found = none_found = 0
    
    for sid in check_ids:
        cp = os.path.join(video_dir, f'{sid}_color.mp4')
        dp = os.path.join(video_dir, f'{sid}_depth.mp4')
        has_c = os.path.exists(cp)
        has_d = os.path.exists(dp)
        if has_c and has_d:  both_found += 1
        elif has_c:          color_found += 1
        elif has_d:          depth_found += 1
        else:                none_found += 1
    
    print(f'  {n_sample} örnekte:')
    print(f'  Her ikisi var: {both_found} ({both_found/n_sample*100:.1f}%)')
    print(f'  Sadece color:  {color_found}')
    print(f'  Sadece depth:  {depth_found}')
    print(f'  Hiçbiri yok:   {none_found}')

print('TRAIN video kontrolü (200 random örnek):')
check_video_availability(train_df, TRAIN_DIR)
print('\nVAL video kontrolü (200 random örnek):')
check_video_availability(val_df, VAL_DIR)

In [ ]:
# ═══════════════════════════════════════════════════
# MEDİAPİPE FONKSİYONLARI
# ═══════════════════════════════════════════════════
mp_holistic = mp.solutions.holistic

def landmarks_to_vec(results):
    """
    MediaPipe Holistic sonuçlarından 225 boyutlu vektör çıkarır.
    Eksik landmark → 0.0 ile doldurulur.
    Dönen: np.ndarray (225,)
    """
    vec = []

    # Sol el: 21 × (x, y, z) = 63
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            vec.extend([lm.x, lm.y, lm.z])
    else:
        vec.extend([0.0] * 63)

    # Sağ el: 21 × (x, y, z) = 63
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            vec.extend([lm.x, lm.y, lm.z])
    else:
        vec.extend([0.0] * 63)

    # Pose: 33 × (x, y, z) = 99
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            vec.extend([lm.x, lm.y, lm.z])
    else:
        vec.extend([0.0] * 99)

    return np.array(vec, dtype=np.float32)


def video_to_landmarks(video_path, max_frames=MAX_FRAMES, img_size=IMG_SIZE):
    """
    Bir video dosyasından max_frames frame seçer,
    her frame için MediaPipe ile landmark vektörü çıkarır.
    Dönen: np.ndarray (max_frames, 225)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return np.zeros((max_frames, SINGLE_LM_DIM), dtype=np.float32)

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((max_frames, SINGLE_LM_DIM), dtype=np.float32)

    # Eşit aralıklı frame indexleri seç
    target_indices = set(np.linspace(0, total - 1, num=max_frames, dtype=int).tolist())
    grabbed = {}
    fid = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if fid in target_indices:
            # BGR → RGB, boyut orijinal kalıyor (512×512)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            grabbed[fid] = frame_rgb
        fid += 1
    cap.release()

    sorted_idx = sorted(target_indices)
    frames = [
        grabbed.get(i, np.zeros((img_size, img_size, 3), dtype=np.uint8))
        for i in sorted_idx
    ]

    # MediaPipe ile her frame'i işle
    # static_image_mode=True: her frame bağımsız, daha doğru
    # model_complexity=2: en yüksek doğruluk
    landmarks = []
    with mp_holistic.Holistic(
        static_image_mode=True,
        model_complexity=2,
        enable_segmentation=False,
        min_detection_confidence=0.3,
        min_tracking_confidence=0.3
    ) as holistic:
        for frame in frames:
            results = holistic.process(frame)
            landmarks.append(landmarks_to_vec(results))

    # Gerekirse son frame ile doldur
    while len(landmarks) < max_frames:
        landmarks.append(
            landmarks[-1].copy() if landmarks
            else np.zeros(SINGLE_LM_DIM, dtype=np.float32)
        )

    return np.array(landmarks[:max_frames], dtype=np.float32)  # (16, 225)


def process_sample(video_dir, sample_id):
    """
    Bir sample için color + depth landmark'larını çıkarır ve concat eder.
    Dönen: (np.ndarray (16, 450), has_color: bool, has_depth: bool)
    """
    color_path = os.path.join(video_dir, f'{sample_id}_color.mp4')
    depth_path = os.path.join(video_dir, f'{sample_id}_depth.mp4')

    has_color = os.path.exists(color_path)
    has_depth = os.path.exists(depth_path)

    color_lm = video_to_landmarks(color_path) if has_color else np.zeros((MAX_FRAMES, SINGLE_LM_DIM), dtype=np.float32)
    depth_lm = video_to_landmarks(depth_path) if has_depth else np.zeros((MAX_FRAMES, SINGLE_LM_DIM), dtype=np.float32)

    combined = np.concatenate([color_lm, depth_lm], axis=-1)  # (16, 450)
    return combined, has_color, has_depth


print('Fonksiyonlar hazır')

In [ ]:
# ═══════════════════════════════════════════════════
# HIZLI TEST: 5 örnek
# ═══════════════════════════════════════════════════
print('Test başlıyor (5 örnek)...')
t0 = time.time()

for sid in train_df['sample_id'].iloc[:5]:
    arr, hc, hd = process_sample(TRAIN_DIR, sid)
    hand_detected = np.count_nonzero(arr[:, :126]) > 0  # color el landmark'ları
    print(f'  {sid}: shape={arr.shape}, color={hc}, depth={hd}, '
          f'el_algilandi={hand_detected}, nonzero={np.count_nonzero(arr)}')

print(f'\n5 örnek için geçen süre: {time.time()-t0:.1f}s')
print(f'Tahmini toplam süre: {(time.time()-t0)/5 * len(train_df) / 60:.0f} dakika')

In [ ]:
# ═══════════════════════════════════════════════════
# ANA EXTRACTION FONKSIYONU (resume destekli)
# ═══════════════════════════════════════════════════
def run_extraction(df, video_dir, cache_dir, split_name):
    """
    Tüm dataset için landmark extraction çalıştırır.
    Zaten işlenmiş dosyaları ATLAR → Colab kopsa bile kaldığı yerden devam eder.
    """
    done = skipped = missing_both = error_count = 0
    start = time.time()

    pbar = tqdm(df.iterrows(), total=len(df), desc=split_name)

    for i, row in pbar:
        sid        = str(row['sample_id'])
        cache_path = os.path.join(cache_dir, f'{sid}.npy')

        # Cache zaten varsa atla (resume)
        if os.path.exists(cache_path):
            skipped += 1
            pbar.set_postfix(done=done, skip=skipped, miss=missing_both, err=error_count)
            continue

        try:
            arr, has_color, has_depth = process_sample(video_dir, sid)

            if not has_color and not has_depth:
                missing_both += 1

            # Her halükarda kaydet (sıfır olsa bile label eşleşmesi korunsun)
            np.save(cache_path, arr)
            done += 1

        except Exception as e:
            error_count += 1
            print(f'\nHATA [{sid}]: {e}')

        pbar.set_postfix(done=done, skip=skipped, miss=missing_both, err=error_count)

        # Her 1000 örnekte özet yaz
        if (done + skipped) % 1000 == 0 and (done + skipped) > 0:
            elapsed  = time.time() - start
            rate     = (done + skipped) / elapsed
            remaining = (len(df) - i - 1) / max(rate, 1)
            print(f'\n[{i+1}/{len(df)}] done={done} skip={skipped} miss={missing_both} '
                  f'| hız={rate:.1f}/s | ~{remaining/60:.0f} dk kaldı')

    elapsed = time.time() - start
    print(f'\n{"="*50}')
    print(f'{split_name} TAMAMLANDI')
    print(f'  İşlenen: {done}, Atlanan (zaten vardı): {skipped}')
    print(f'  Her iki video da eksik olan: {missing_both}')
    print(f'  Hata: {error_count}')
    print(f'  Toplam süre: {elapsed/60:.1f} dakika')
    print(f'{'="*50}')

print('Extraction fonksiyonu hazır')

In [ ]:
# ═══════════════════════════════════════════════════
# TRAIN EXTRACTION — ÇALIŞTIR
# Not: Yarıda kesilirse tekrar çalıştır, kaldığı yerden devam eder
# ═══════════════════════════════════════════════════
run_extraction(train_df, TRAIN_DIR, TRAIN_CACHE, 'TRAIN')

In [ ]:
# ═══════════════════════════════════════════════════
# VALIDATION EXTRACTION
# ═══════════════════════════════════════════════════
run_extraction(val_df, VAL_DIR, VAL_CACHE, 'VALIDATION')

In [ ]:
# ═══════════════════════════════════════════════════
# KALİTE KONTROLÜ
# ═══════════════════════════════════════════════════
from collections import Counter

def quality_check(df, cache_dir, split_name):
    print(f'\n{"="*55}')
    print(f'{split_name} Kalite Kontrolü')
    print(f'{"="*55}')

    shapes      = Counter()
    all_zero    = 0
    no_hand     = 0   # el landmark'ı hiç algılanmamış
    missing     = []
    bad_files   = []

    for sid in tqdm(df['sample_id'], desc='Kontrol', leave=False):
        p = os.path.join(cache_dir, f'{sid}.npy')
        if not os.path.exists(p):
            missing.append(sid)
            continue
        try:
            arr = np.load(p)
            shapes[str(arr.shape)] += 1
            if np.count_nonzero(arr) == 0:
                all_zero += 1
            # El landmark'ları: ilk 126 eleman (color sol+sağ el)
            if np.count_nonzero(arr[:, :126]) == 0:
                no_hand += 1
        except Exception:
            bad_files.append(sid)

    total      = len(df)
    correct    = shapes.get('(16, 450)', 0)
    pct        = correct / total * 100

    print(f'Shape dağılımı:         {dict(shapes)}')
    print(f'Doğru shape (16,450):   {correct} / {total} ({pct:.1f}%)')
    print(f'Eksik cache:            {len(missing)}')
    print(f'Bozuk dosya:            {len(bad_files)}')
    print(f'Tamamen sıfır:          {all_zero} (her iki video da yoktu)')
    print(f'El hiç algılanmamış:    {no_hand} ({no_hand/total*100:.1f}%)')

    if correct == total:
        print('✅ Tüm cache dosyaları doğru!')
    elif len(missing) > 0:
        print(f'⚠️  {len(missing)} eksik var — extraction hücresini tekrar çalıştır')

    return correct == total

train_ok = quality_check(train_df, TRAIN_CACHE, 'TRAIN')
val_ok   = quality_check(val_df,   VAL_CACHE,   'VALIDATION')

In [ ]:
# ═══════════════════════════════════════════════════
# PACKED DATASET — eğitim için tek seferde yüklenebilir
# ═══════════════════════════════════════════════════
def pack_dataset(df, cache_dir, prefix):
    X_list, y_list, bad = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Packing {prefix}'):
        sid   = str(row['sample_id'])
        label = int(row['label'])
        p     = os.path.join(cache_dir, f'{sid}.npy')

        if not os.path.exists(p):
            bad.append(sid)
            continue

        arr = np.load(p)
        if arr.shape != (MAX_FRAMES, FINAL_DIM):
            bad.append(sid)
            continue

        X_list.append(arr)
        y_list.append(label)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)

    np.save(f'{PACKED_DIR}/{prefix}_X.npy', X)
    np.save(f'{PACKED_DIR}/{prefix}_y.npy', y)

    print(f'{prefix}: X={X.shape}, y={y.shape}')
    if bad:
        print(f'  Atlandı: {len(bad)} örnek')
    return X, y

X_train, y_train = pack_dataset(train_df, TRAIN_CACHE, 'train')
X_val,   y_val   = pack_dataset(val_df,   VAL_CACHE,   'val')

# Meta dosyası
meta = (
    f'X_train: {X_train.shape}\n'
    f'y_train: {y_train.shape}\n'
    f'X_val: {X_val.shape}\n'
    f'y_val: {y_val.shape}\n'
    f'num_classes: {len(np.unique(y_train))}\n'
    f'seq_len: {MAX_FRAMES}\n'
    f'feat_dim: {FINAL_DIM}\n'
    f'layout: color[sol_el(63)+sag_el(63)+pose(99)] + depth[sol_el(63)+sag_el(63)+pose(99)]\n'
)
with open(f'{PACKED_DIR}/meta.txt', 'w') as f:
    f.write(meta)

print('\n✅ Packed dataset kaydedildi:')
print(f'  {PACKED_DIR}/train_X.npy  — {X_train.nbytes/1e6:.0f} MB')
print(f'  {PACKED_DIR}/val_X.npy    — {X_val.nbytes/1e6:.0f} MB')
print('\nSonraki adım: 02_model_training.ipynb')